
## Troubleshooting guide:

<!-- # reading the kafka topic to inspect the Debezium messages:
docker exec -it kafka /opt/kafka/bin/kafka-console-consumer.sh `
  --bootstrap-server localhost:9092 `
  --topic retail.MichaelN_Retail.dbo.RetailCustomer `
  --from-beginning -->

In [0]:


%sql

-- # this is the SCD type 2 with all the neccessary updates


SELECT
    CustomerID,
    Age,
    Gender,
    Location,
    SubscriptionStatus,
    PreviousPurchases,
    FrequencyOfPurchases,
    ChangeTimestamp,
    ValidFrom,
    ValidTo,
    IsCurrent
FROM delta.`/Volumes`  -- hidden for privacy
-- WHERE CustomerID = 8
ORDER BY ValidFrom;

CustomerID,Age,Gender,Location,SubscriptionStatus,PreviousPurchases,FrequencyOfPurchases,ChangeTimestamp,ValidFrom,ValidTo,IsCurrent
8,27,Male,Florida,Yes,19,Weekly,2026-08-21 16:07:59,2026-08-21 16:07:59,2026-08-21T17:09:03.000Z,false
8,27,Male,Nevada,Yes,19,Weekly,2026-08-21 17:09:03,2026-08-21 17:09:03,2026-08-21T17:28:12.000Z,false
8,27,Male,North Carolina,Yes,19,Weekly,2026-08-21 17:28:12,2026-08-21 17:28:12,2026-08-25T12:44:57.000Z,false
8,27,Male,New York,Yes,19,Weekly,2026-08-25 12:44:57,2026-08-25 12:44:57,null,true
1,64,Male,New York,Yes,16,Fortnightly,2026-08-25 13:07:36,2026-08-25 13:07:36,2026-08-25T13:45:20.000Z,false
1,64,Male,Delaware,Yes,16,Fortnightly,2026-08-25 13:45:20,2026-08-25 13:45:20,2026-08-25T13:52:36.000Z,false
1,64,Male,Montana,Yes,16,Fortnightly,2026-08-25 13:52:36,2026-08-25 13:52:36,2026-08-25T14:09:15.000Z,false
1,64,Male,Ohio,Yes,16,Fortnightly,2026-08-25 14:09:15,2026-08-25 14:09:15,null,true
2,19,Male,Ohio,Yes,2,Fortnightly,2026-08-25 14:18:16,2026-08-25 14:18:16,null,true
5,45,Male,Arizona,Yes,31,Annually,2026-08-25 14:26:13,2026-08-25 14:26:13,2026-08-25T19:05:36.000Z,false


In [0]:
%sql

-- Gold view with only the current records for each customer 

CREATE OR REPLACE TEMP VIEW gold_current_customer AS

SELECT
    CustomerID,
    Age,
    Gender,
    Location,
    SubscriptionStatus,
    PreviousPurchases,
    FrequencyOfPurchases,
    ChangeTimestamp,
    ValidFrom,
    ValidTo,
    IsCurrent
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY CustomerID
            ORDER BY ValidFrom DESC
        ) AS rn
    FROM delta.`/Volumes/`  -- hidden for privacy
)
WHERE rn = 1  AND IsCurrent = true;


select *
from gold_current_customer


CustomerID,Age,Gender,Location,SubscriptionStatus,PreviousPurchases,FrequencyOfPurchases,ChangeTimestamp,ValidFrom,ValidTo,IsCurrent
1,64,Male,Ohio,Yes,16,Fortnightly,2026-08-25 14:09:15,2026-08-25 14:09:15,null,true
2,19,Male,Ohio,Yes,2,Fortnightly,2026-08-25 14:18:16,2026-08-25 14:18:16,null,true
5,45,Male,Wyoming,Yes,31,Annually,2026-08-25 19:05:36,2026-08-25 19:05:36,null,true
8,27,Male,New York,Yes,19,Weekly,2026-08-25 12:44:57,2026-08-25 12:44:57,null,true
3341,20,Female,California,No,35,Fortnightly,2026-08-25 19:28:10,2026-08-25 19:28:10,null,true


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, lit

# Silver SCD2 table

SILVER_SCD2_PATH = "/Volu" # hidden for privacy

if not DeltaTable.isDeltaTable(spark, SILVER_SCD2_PATH):

    silver_initial = (
        after_df
        .select(
            "CustomerID",
            "Age",
            "Gender",
            "Location",
            "SubscriptionStatus",
            "PreviousPurchases",
            "FrequencyOfPurchases",
            "ChangeTimestamp"
        )
        .withColumn("ValidFrom", col("ChangeTimestamp"))
        .withColumn("ValidTo", lit(None).cast("timestamp"))
        .withColumn("IsCurrent", lit(True))
    )

    (
        silver_initial.write
        .format("delta")
        .mode("append")
        .save(SILVER_SCD2_PATH)
    )

    print("Silver SCD2 table created.")

else:

    silver_table = DeltaTable.forPath(spark, SILVER_SCD2_PATH)

    # Close changed current records
    (
        silver_table.alias("target")
        .merge(
            after_df.alias("source"),
            """
            target.CustomerID = source.CustomerID
            AND target.IsCurrent = true
            """
        )
        .whenMatchedUpdate(
            condition="""
                NOT(target.Age <=> source.Age)
                OR NOT(target.Gender <=> source.Gender)
                OR NOT(target.Location <=> source.Location)
                OR NOT(target.SubscriptionStatus <=> source.SubscriptionStatus)
                OR NOT(target.PreviousPurchases <=> source.PreviousPurchases)
                OR NOT(target.FrequencyOfPurchases <=> source.FrequencyOfPurchases)
            """,
            set={
                "ValidTo": "source.ChangeTimestamp",
                "IsCurrent": "false"
            }
        )
        .execute()
    )

    # Insert new current versions
    new_versions = (
        after_df.alias("source")
        .join(
            silver_table.toDF().alias("target"),
            (
                (col("source.CustomerID") == col("target.CustomerID")) &
                (col("target.IsCurrent") == lit(True))
            ),
            "left"
        )
        .filter(col("target.CustomerID").isNull())
        .select(
            col("source.CustomerID"),
            col("source.Age"),
            col("source.Gender"),
            col("source.Location"),
            col("source.SubscriptionStatus"),
            col("source.PreviousPurchases"),
            col("source.FrequencyOfPurchases"),
            col("source.ChangeTimestamp"),
            col("source.ChangeTimestamp").alias("ValidFrom")
        )
        .withColumn("ValidTo", lit(None).cast("timestamp"))
        .withColumn("IsCurrent", lit(True))
    )

    (
        new_versions.write
        .format("delta")
        .mode("append")
        .save(SILVER_SCD2_PATH)
    )

    print("SCD Type 2 update applied successfully.")






# display(
#     spark.read
#     .format("delta")
#     .load(SILVER_SCD2_PATH)
#     .filter(col("CustomerID") == 8)
#     .orderBy("ValidFrom")
# )

SCD Type 2 update applied successfully.


In [0]:
from pyspark.sql.functions import col, from_unixtime, row_number
from pyspark.sql.window import Window

after_df = (
    cdc_df
    .filter(
        (col("op").isin("c", "u", "r")) &
        (col("after").isNotNull())
    )
    .select(
        col("offset"),
        col("after.CustomerID").alias("CustomerID"),
        col("after.Age").alias("Age"),
        col("after.Gender").alias("Gender"),
        col("after.Location").alias("Location"),
        col("after.SubscriptionStatus").alias("SubscriptionStatus"),
        col("after.PreviousPurchases").alias("PreviousPurchases"),
        col("after.FrequencyOfPurchases").alias("FrequencyOfPurchases"),
        col("op"),
        from_unixtime(col("cdc_ts_ms") / 1000).alias("ChangeTimestamp")
    )
)

# Keep only the latest CDC event for each customer in this batch
latest_change_window = Window.partitionBy("CustomerID").orderBy(
    col("ChangeTimestamp").desc(),
    col("offset").desc()
)

after_df = (
    after_df
    .withColumn("rn", row_number().over(latest_change_window))
    .filter(col("rn") == 1)
    .drop("rn")
)

display(after_df)

offset,CustomerID,Age,Gender,Location,SubscriptionStatus,PreviousPurchases,FrequencyOfPurchases,op,ChangeTimestamp
3250,3341,20,Female,California,No,35,Fortnightly,c,2026-08-25 19:28:10


In [0]:
from pyspark.sql.functions import col, from_unixtime

BRONZE_PATH = "/Volumes/h.json" # hidden for privacy

bronze_df = (spark.read.option("multiline", "true").json(BRONZE_PATH))



# Change data capture dataframe creation 


cdc_df = (bronze_df.select(
        col("offset"),
        col("partition"),
        col("topic"),
        col("value.payload.before").alias("before"),
        col("value.payload.after").alias("after"),
        col("value.payload.op").alias("op"),
        col("value.payload.source.change_lsn").alias("change_lsn"),
        col("value.payload.source.commit_lsn").alias("commit_lsn"),
        col("value.payload.source.snapshot").alias("snapshot"),
        col("value.payload.ts_ms").alias("cdc_ts_ms")
    )
)

display(cdc_df)

offset,partition,topic,before,after,op,change_lsn,commit_lsn,snapshot,cdc_ts_ms
3250,0,retail.MichaelN_Retail.dbo.RetailCustomer,null,"List(20, 3341, Fortnightly, Female, California, 35, No)",c,00000035:0001ea68:0002,00000035:0001ea68:0003,false,1787686090983
